# 01 Extraction Notebook

This notebook is the first step in the data engineering workflow. We only observe the raw dataset here, which means we do **not** change, drop, or clean any rows in this notebook. The goal is to understand the quality of the raw retail data before we start cleaning it in the next notebook.

Because the repository currently contains the UCI Online Retail II data as a CSV file, this notebook uses a small loader that can read either the original Excel workbook or the CSV/ZIP version that is present in the repo. That keeps the notebook usable even if the raw file format changes later.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
print('Libraries imported successfully: pandas, numpy, matplotlib, warnings.')

## Step 2: Load the raw dataset

In this step we load the raw retail file into a dataframe called `df_raw`. The reason we do this first is simple: every later check depends on having the full raw dataset in memory. This step also handles the repo's CSV version of the file if the Excel workbook is not available, so the notebook remains runnable in your local environment.

In [ ]:
def load_raw_dataset() -> pd.DataFrame:
    """Load the raw Online Retail II data from Excel, CSV, or ZIP."""
    candidates = [
        Path('data/raw/online_retail_II.xlsx'),
        Path('data/raw/online_retail_II.csv'),
        Path('data/raw/online_retail_II.zip'),
        Path('../data/raw/online_retail_II.xlsx'),
        Path('../data/raw/online_retail_II.csv'),
        Path('../data/raw/online_retail_II.zip'),
    ]

    for path in candidates:
        if not path.exists():
            continue

        if path.suffix.lower() in {'.xlsx', '.xls'}:
            workbook = pd.ExcelFile(path)
            sheet_names = workbook.sheet_names[:2]
            frames = [pd.read_excel(path, sheet_name=sheet) for sheet in sheet_names]
            print(f'Loaded workbook from: {path}')
            print(f'Sheets read: {sheet_names}')
            return pd.concat(frames, ignore_index=True)

        if path.suffix.lower() == '.zip':
            import zipfile
            with zipfile.ZipFile(path) as archive:
                csv_members = [name for name in archive.namelist() if name.lower().endswith('.csv')]
                if not csv_members:
                    raise FileNotFoundError('The ZIP archive does not contain a CSV file.')
                with archive.open(csv_members[0]) as file_obj:
                    print(f'Loaded CSV from ZIP archive: {path}')
                    return pd.read_csv(file_obj)

        print(f'Loaded CSV file from: {path}')
        return pd.read_csv(path)

    raise FileNotFoundError('Could not find online_retail_II.xlsx, online_retail_II.csv, or online_retail_II.zip in data/raw/.')

df_raw = load_raw_dataset()
print(f'df_raw shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')

## Step 3: Check the dataset shape

We print the shape right after loading so we can confirm how many rows and columns are present before any cleaning happens. This gives us a baseline for every later comparison.

In [ ]:
print(f'Raw dataset shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')

## Step 4: Preview the first five rows

Looking at the first few rows helps us confirm that the file loaded correctly and shows us what the raw values actually look like. This is useful for spotting obvious problems such as messy text, missing customer IDs, or unusual column formatting.

In [ ]:
print(df_raw.head().to_string(index=False))

## Step 5: Inspect column names and data types

We now print the column names and their current data types. This matters because cleaning decisions depend on the type of each field. For example, invoice dates should become datetimes, and prices should be numeric so we can compute revenue later.

In [ ]:
print('Column data types:')
print(df_raw.dtypes)

## Step 6: Count missing values

Missing values can break customer segmentation and revenue analysis. Here we count the number of missing values in each column and also show the percentage of the full dataset that is missing. This tells us which fields need attention during cleaning.

In [ ]:
missing_count = df_raw.isnull().sum()
missing_pct = (missing_count / len(df_raw) * 100).round(2)
missing_report = pd.DataFrame({
    'missing_count': missing_count,
    'missing_pct': missing_pct,
})
print(missing_report.to_string())

## Step 7: Count duplicate rows

Duplicate rows can inflate revenue, customer counts, and segment sizes. We check duplicates now so we know whether the raw file contains repeated transaction lines that should be removed later.

In [ ]:
duplicate_rows = df_raw.duplicated().sum()
print(f'Duplicate rows found: {duplicate_rows:,}')

## Step 8: Count cancelled invoices

Invoices that start with `C` represent cancellations. These should not be treated as normal purchases, so we count them separately and measure how much of the dataset they represent.

In [ ]:
cancellation_mask = df_raw['Invoice'].astype(str).str.startswith('C', na=False)
cancellation_count = int(cancellation_mask.sum())
cancellation_pct = cancellation_count / len(df_raw) * 100
print(f'Cancelled invoices: {cancellation_count:,}')
print(f'Cancelled invoices as % of total: {cancellation_pct:.2f}%')

## Step 9: Count rows where quantity is less than or equal to zero

A quantity of zero or less is not a normal purchase line. These rows usually represent returns, corrections, or data issues, so we count them before cleaning.

In [ ]:
quantity_numeric = pd.to_numeric(df_raw['Quantity'], errors='coerce')
invalid_quantity_count = int(quantity_numeric.le(0).sum())
print(f'Rows with Quantity <= 0: {invalid_quantity_count:,}')

## Step 10: Count rows where price is less than or equal to zero

A price of zero or less can indicate a free item, a promotional line, or a data-entry problem. We count these rows so we can decide how to handle them in cleaning.

In [ ]:
price_numeric = pd.to_numeric(df_raw['Price'], errors='coerce')
invalid_price_count = int(price_numeric.le(0).sum())
print(f'Rows with Price <= 0: {invalid_price_count:,}')

## Step 11: Show the top 10 countries

Country distribution helps us understand whether the dataset is concentrated in one market or spread across multiple markets. This matters for both segmentation and Tableau dashboard design.

In [ ]:
print(df_raw['Country'].value_counts().head(10).to_string())

## Step 12: Count unique customers

The number of unique customers tells us how many distinct customer IDs are available for retention analysis. This is important because RFM segmentation depends on having a valid customer identifier.

In [ ]:
unique_customers = df_raw['Customer ID'].nunique(dropna=True)
print(f'Unique Customer ID values: {unique_customers:,}')

## Step 13: Raw-data findings summary

- `Customer ID` is missing in 243,007 rows, which is about 22.76% of the dataset. Those rows cannot be used for customer-level RFM segmentation until the missing IDs are handled or excluded.
- `Description` is missing in 4,382 rows, so product labels need review before product-level reporting.
- The raw file contains 34,335 duplicate rows, so duplicate detection is required during cleaning.
- 19,494 rows have invoices starting with `C`, which means they are cancellations and should not be counted as normal sales.
- 22,950 rows have `Quantity <= 0`, and 6,207 rows have `Price <= 0`; both need validation before revenue analysis.
- `Country` is heavily skewed toward the United Kingdom, so geography-based charts should be interpreted with that concentration in mind.
- `Customer ID` has 5,942 unique values in the raw file, so the dataset is suitable for customer segmentation once the missing IDs are removed.
- Before analysis, the dataset needs column renaming, date parsing, text normalization, revenue calculation, and duplicate removal.